# Quickstart: is a pre-treatment signal about the drug, or about growth fitness?

This notebook reruns the core analysis of the paper on the small example file `data/example/resistrace_pca.npz`
(7 MB, included in the repository), so no download is needed. It contains the 32,642 labelled pre-treatment cells
of the ReSisTrace experiment (Dai et al., *Nat Commun* 2024; GEO GSE223003): 50 principal components of gene
expression, the fate label (1 = the cell's sister was recovered after treatment), the lineage family, sample and
arm of each cell, and the 3,477 pre-treatment sister pairs. `scripts/export_example_data.py` shows how it was made.

Steps: (1) predict each cell's sister fate within each arm, (2) compare each drug-arm model with a *fitness model*
trained on the untreated control arm, (3) estimate how much is lost because the label belongs to the sister.
Run time is about two minutes on a laptop CPU.

In [1]:
import numpy as np
from sklearn.metrics import roc_auc_score
from persist.fitadj import grouped_oof, fitness_adjusted_report, sister_concordance, sister_ceiling

d = np.load("../data/example/resistrace_pca.npz")
X, y, family, sample, arm = d["X_pca"], d["y"], d["family"], d["sample"], d["arm"]
print(f"{len(y)} cells, {y.sum()} pre-resistant, arms: {sorted(set(arm.tolist()))}")

32642 cells, 3102 pre-resistant, arms: ['Carboplatin', 'Control', 'NK', 'Olaparib']


## 1. The pre-treatment state predicts the sister's fate

Logistic regression on the 50 principal components, five-fold cross-validation within each arm. Whole families
stay on one side of every split, so a model is never tested on a relative of a cell it was trained on.
The paper reports 0.747 (carboplatin), 0.744 (olaparib), 0.723 (NK) and 0.708 (control).

In [2]:
scores = {}
for a in ["Carboplatin", "Olaparib", "NK", "Control"]:
    m = arm == a
    scores[a] = grouped_oof(X[m], y[m], family[m])
    print(f"{a:12s} AUROC {roc_auc_score(y[m], scores[a]):.3f}")

Carboplatin  AUROC 0.747


Olaparib     AUROC 0.744


NK           AUROC 0.723


Control      AUROC 0.708


## 2. Most of the signal is shared with the untreated arm

`fitness_adjusted_report` trains a *fitness model* on the control arm, where a sister is recovered without any
drug, and scores each drug arm with it. `delta_size_matched` is the drug-specific gain: how much better the
drug-arm model does than the fitness model when both are trained on the same number of cells.
Here we use 3 training subsamples and 100 bootstrap replicates to keep it fast (the paper uses 10 and 1,000),
so the gains can differ from the paper's 0.014, 0.025 and 0.017 in the third decimal.
The paper reports fitness-model AUROCs of 0.725, 0.713 and 0.697.

In [3]:
report = fitness_adjusted_report(X, y, family, arm, sample, control="Control", n_subsamples=3, n_boot=100)
report[["arm", "auroc_drug", "auroc_fitness", "delta_size_matched", "delta_lo", "delta_hi"]].round(3)

,arm,auroc_drug,auroc_fitness,delta_size_matched,delta_lo,delta_hi
0,Carboplatin,0.747,0.725,0.017,0.002,0.028
1,NK,0.723,0.697,0.013,0.000,0.028
2,Olaparib,0.744,0.713,0.024,0.010,0.039


The fitness model, which never saw a drug, comes within 0.02–0.03 AUROC of each drug-arm model. The interval
shown is the fast one that only resamples the evaluation; the paper shows by simulation that it is too narrow and
reports a slower interval that refits the models (`scripts/28_delta_refit_bootstrap.py`), which does not
reliably exclude zero.

## 3. The sister ceiling

The label belongs to the treated sister, not to the sequenced cell. From how similar the model's scores are
between pre-treatment sisters, `sister_ceiling` estimates the AUROC the same score would reach on the treated
sister itself, assuming 10% of pre-sensitive labels are wrong (the paper reports about 0.84 for the drug arms).

In [4]:
pairs = d["sister_pairs"]
all_scores = np.empty(len(y))
for a, s in scores.items():
    all_scores[arm == a] = s
for a in ["Carboplatin", "Olaparib", "NK"]:
    m = arm == a
    in_arm = m[pairs[:, 0]] & m[pairs[:, 1]]
    c = sister_concordance(all_scores, pairs[in_arm])
    out = sister_ceiling(roc_auc_score(y[m], scores[a]), c, prevalence=y[m].mean())
    print(a, {k: round(float(v), 3) for k, v in out.items()})

Carboplatin {'rho_obs': 0.497, 'rho_fate': 0.695, 'saturated': 0.0, 'auroc_if_sister_seen': 0.838, 'sister_cost': 0.091}


Olaparib {'rho_obs': 0.499, 'rho_fate': 0.709, 'saturated': 0.0, 'auroc_if_sister_seen': 0.839, 'sister_cost': 0.096}


NK {'rho_obs': 0.448, 'rho_fate': 0.706, 'saturated': 0.0, 'auroc_if_sister_seen': 0.842, 'sister_cost': 0.119}


## Next steps

- Use `persist.fitadj` on your own lineage-tracing data: any representation `X`, labels `y`, families and arms.
- `scripts/run_all.sh` reruns every analysis in the paper from the raw public data.